### CNN FOR CIFAR10 Multi-class Classification

In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim

In [2]:
import torchvision 
from torchvision.datasets import CIFAR10

### Dataset and DataLoader

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms 

In [4]:
transform=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ]
)

In [5]:
trainset=CIFAR10(root="./data",train=True,download=False,transform=transform)
testset=CIFAR10(root="./data",train=False,download=False,transform=transform)

c:\Users\HP\anaconda3\envs\deep_learning\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [7]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [53]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

### CNN

In [54]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel=>2,stride=>2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1) #flatenning
        x=self.fc_layers(x)

        return x

In [55]:
model=CNN()

In [56]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training the CNN

In [57]:
epochs=10


for epoch in range(epochs):
    training_loss=0.0

    for images,labels in trainloader:


        optimizer.zero_grad()

        output=model.forward(images)  #Forward Propogation
        loss=criteria(output,labels) #Loss Fnx
        loss.backward() #Backward Propogation
        optimizer.step()

        training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} and loss={training_loss/len(trainloader)}")

1/10 and loss=1.3644054293480066
2/10 and loss=0.9260048062142814
3/10 and loss=0.7401248727689314
4/10 and loss=0.6074040679218214
5/10 and loss=0.5043819353670416
6/10 and loss=0.4057088065749544
7/10 and loss=0.3185729509043267
8/10 and loss=0.2500592357648151
9/10 and loss=0.19000061114540184
10/10 and loss=0.15156855239578143


### Evaluate our model

In [58]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

In [59]:
print(f"Accuracy is {correct_labels/total_labels*100}")

Accuracy is 75.49
